# <font style="color:blue">Project 3: Kaggle Competition - Semantic Segmentation</font>

#### Maximum Points: 100

<div>
    <table>
        <tr><td><h3>Sr. no.</h3></td> <td><h3>Section</h3></td> <td><h3>Points</h3></td> </tr>
        <tr><td><h3>1</h3></td> <td><h3>1.1. Dataset Class</h3></td> <td><h3>7</h3></td> </tr>
        <tr><td><h3>2</h3></td> <td><h3>1.2. Visualize dataset</h3></td> <td><h3>3</h3></td> </tr>
        <tr><td><h3>3</h3></td> <td><h3>2. Evaluation Metrics</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>4</h3></td> <td><h3>3. Model</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>5</h3></td> <td><h3>4.1. Train</h3></td> <td><h3>7</h3></td> </tr>
        <tr><td><h3>6</h3></td> <td><h3>4.2. Inference</h3></td> <td><h3>3</h3></td> </tr>
        <tr><td><h3>7</h3></td> <td><h3>5. Prepare Submission CSV</h3></td><td><h3>10</h3></td> </tr>
        <tr><td><h3>8</h3></td> <td><h3>6. Kaggle Profile Link</h3></td> <td><h3>50</h3></td> </tr>
    </table>
</div>

---

**In this project, you have participated in the Kaggle competition, and also submit the notebook and othe code in the course lab.**

**This Kaggle competition is a semantic segmentation challenge.**

<h2>Dataset Description </h2>
<p>The dataset consists of 3,269 images in 12 classes (including background). All images were taken from drones in a variety of scales. Samples are shown below:
<img src="https://www.dropbox.com/scl/fi/pswwraz1cc9srd9d4hxm3/data_montage.jpg?rlkey=074v9mc32et70ijl0dz3y0rvs&dl=1" width="800" height="800">
<p>The data was splitted into public train set and private test set which is used for evaluation of submissions. You can split public subset into train and validation sets yourself.
Images are named with a unique <code>ImageId</code>. </p>
<p> You should segment and classify the images in the test set.</p>
<p>The dataset consists of landscape images taken from drones in a variety of scales.</p>

**The notebook is divided into sections. You have to write code, as mention in the section.  For other helper functions, you can write `.py` files and import them in the notebook. You have to submit the notebook along with `.py` files. Your submitted code must be runnable without any bug.**

In [ ]:
!pip install -qU livelossplot 

In [ ]:
# Importing  libraries
import gc
import os
import shutil
import zipfile
import warnings
from glob import glob
from dataclasses import dataclass

import cv2
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import Union

import torch
import torch.nn as nn
from torch.cuda import amp
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision.models import segmentation

from torchmetrics import MeanMetric
from torchmetrics.classification import MulticlassAccuracy

from torchinfo import summary

# For data augmentation and preprocessing.
import albumentations as A
from albumentations.pytorch import ToTensorV2

# For plotting training and validation metrics.
from livelossplot import PlotLosses
from livelossplot.outputs import MatplotlibPlot, ExtremaPrinter

# To filter UserWarning.
warnings.filterwarnings("ignore", category=UserWarning)

## Data Validation

In [ ]:
# TO check whether all the images contain jpg format
path_img = '/kaggle/input/open-cv-py-torch-segmentation-project-round-3/imgs/imgs/*.*'
other_files = [i for i in glob(path_img) if not i.lower().endswith('.jpg')]
print(len(other_files))

In [ ]:
# TO check whether all the masks contain png format
path_mask = '/kaggle/input/open-cv-py-torch-segmentation-project-round-3/masks/masks/*.*'
other_files = [i for i in glob(path_mask) if not i.lower().endswith('.png')]
print(len(other_files))

# **Configuration setup**

In [ ]:
def get_default_device():
    gpu_available = torch.cuda.is_available()
    return torch.device('cuda' if gpu_available else 'cpu'), gpu_available

In [ ]:
get_default_device()

In [ ]:
def seed_everything(seed_value):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Training and Dataset configurations


In [ ]:
# paths 
root_dir = '/kaggle/working/'
train_img_datasets = os.path.join(root_dir, 'datasets/train/images')
train_mask_datasets = os.path.join(root_dir, 'datasets/train/masks')
valid_img_datasets = os.path.join(root_dir, 'datasets/valid/images')
valid_mask_datasets = os.path.join(root_dir, 'datasets/valid/masks')
test_img_datasets = os.path.join(root_dir, 'datasets/test/images')
model_checkpoint = os.path.join(root_dir, 'model_checkpoint/DeepLabv3')

In [ ]:
# creating folders in the working directory
os.makedirs(train_img_datasets, exist_ok = True)
os.makedirs(train_mask_datasets, exist_ok = True)
os.makedirs(valid_img_datasets, exist_ok = True)
os.makedirs(valid_mask_datasets, exist_ok = True)
os.makedirs(test_img_datasets, exist_ok = True)
os.makedirs(model_checkpoint, exist_ok = True)

In [ ]:
@dataclass(frozen=True)
class DatasetConfig:
    NUM_CLASSES: int = 12
    IMG_WIDTH:   int = 512
    IMG_HEIGHT:  int = 384

    DATA_TRAIN_IMAGES: str = os.path.join(train_img_datasets, r"*.jpg")
    DATA_TRAIN_LABELS: str = os.path.join(train_mask_datasets,  r"*.png")
    DATA_VALID_IMAGES: str = os.path.join(valid_img_datasets, r"*.jpg")
    DATA_VALID_LABELS: str = os.path.join(valid_mask_datasets,  r"*.png")
    DATA_TEST_IMAGES:  str = os.path.join(test_img_datasets, r"*.jpg")

@dataclass(frozen=True)
class TrainingConfig:
    BATCH_SIZE:      int = 8
    EPOCHS:          int = 70
    LEARNING_RATE: float = 1e-4
    CHECKPOINT_DIR:  str = os.path.join('model_checkpoint', 'DeepLabv3')
    NUM_WORKERS:     int = os.cpu_count()


@dataclass(frozen=True)
class InferenceConfig:
    BATCH_SIZE:  int = 4
    NUM_BATCHES: int = 4

### Copying images from imgs to train and test

In [ ]:
# reading csv files of train and test files
df_train= pd.read_csv('/kaggle/input/open-cv-py-torch-segmentation-project-round-3/train.csv')
df_test = pd.read_csv('/kaggle/input/open-cv-py-torch-segmentation-project-round-3/test.csv')

In [ ]:
# length of train data
len(df_train)

In [ ]:
# length of test data
len(df_test)

In [ ]:
train_names = df_train.iloc[:,0].tolist()
test_names = df_test.iloc[:,0].tolist()

In [ ]:
# Copying images from imgs folder to specified train directory
for i in train_names:
    name_jpg = str(i) + '.jpg'
    name_png = str(i) + '.png'
    
    src_image = os.path.join('/kaggle/input/open-cv-py-torch-segmentation-project-round-3/imgs/imgs', 
                             name_jpg)
    src_mask = os.path.join('/kaggle/input/open-cv-py-torch-segmentation-project-round-3/masks/masks', 
                            name_png)
    
    dst_image = os.path.join(train_img_datasets, 
                             name_jpg)
    dst_mask = os.path.join(train_mask_datasets, 
                            name_png)
    
    if os.path.exists(src_image):
        shutil.copy2(src_image, dst_image)

    if os.path.exists(src_mask):
        shutil.copy2(src_mask, dst_mask)
    
    

In [ ]:
# Copying images from imgs folder to specified test directory
for i in test_names:
    name_jpg = str(i) + '.jpg'

    src_image = os.path.join('/kaggle/input/open-cv-py-torch-segmentation-project-round-3/imgs/imgs', 
                             name_jpg)
    dst_image = os.path.join(test_img_datasets, 
                             name_jpg)

    if os.path.exists(src_image):
        shutil.copy2(src_image, dst_image)

In [ ]:
# validating the number of files in train folder
print(len(glob(DatasetConfig.DATA_TRAIN_IMAGES)))

In [ ]:
# validating the number of files in test folder
print(len(glob(DatasetConfig.DATA_TEST_IMAGES)))

In [ ]:
# creating train_test_split of training data 
_, valid = train_test_split(train_names,
                            test_size = 0.2,
                            random_state  = 42)
for i in valid:
    name_jpg = str(i) + '.jpg'
    name_png = str(i) + '.png'

    source_img_valid = os.path.join(train_img_datasets, name_jpg)
    dst_img_valid = os.path.join(valid_img_datasets, name_jpg)
        
    source_mask_valid = os.path.join(train_mask_datasets, name_png)
    dst_mask_valid = os.path.join(valid_mask_datasets, name_png)

    if os.path.exists(source_img_valid):
        shutil.move(source_img_valid, dst_img_valid)
        
    if os.path.exists(source_mask_valid):
        shutil.move(source_mask_valid, dst_mask_valid)

In [ ]:
print(f"Train Images: {len(os.listdir(train_img_datasets))}")
print(f"Valid Images: {len(os.listdir(valid_img_datasets))}")
print(f"Train masks: {len(os.listdir(train_mask_datasets))}")
print(f"Valid masks: {len(os.listdir(valid_mask_datasets))}")

# <font style="color:green">1. Data Exploration</font>

In this section, you have to write your custom dataset class and visualize a few images (max five images) and its mask.

## <font style="color:green">1.1. Dataset Class [7 Points]</font>

**In this sub-section, write your custom dataset class.**


**Note that there are not separate validation data, so you will have to create your validation set by dividing train data into train and validation data. Usually, in practice, we do `80:20` ratio for train and validation, respectively.** 

**for example:**

```
class SemSegDataset(Dataset):
    """ Generic Dataset class for semantic segmentation datasets.

        Arguments:
            data_path (string): Path to the dataset folder.
            images_folder (string): Name of the folder containing the images (related to the data_path).
            masks_folder (string): Name of the folder containing the masks (related to the data_path).
            csv_path (string): train or test csv file name
            image_ids (list): List of images.
            train_val_test (string): 'train', 'val' or 'test'
            transforms (callable, optional): A function/transform that inputs a sample
                and returns its transformed version.
            class_names (list, optional): Names of the classes.
            

        Dataset folder structure:
            Folder containing the dataset should look like:
            - data_path
            -- images_folder
            -- masks_folder

            Names of images in the images_folder and masks_folder should be the same for same samples.
    """
```

In [ ]:
# Custom Class for creating training and validation (segmentation) dataset objects.
class CustomSegDataset(Dataset):
    def __init__(self, *, 
                 image_size, num_classes, 
                 image_paths, mask_paths = None, 
                 is_train=False):
        self.image_size  = image_size
        self.image_paths = image_paths
        self.mask_paths  = mask_paths # None in case of test set where only images are available.
        self.num_classes = num_classes
        self.is_train = is_train
        self.transforms = self.setup_transforms()

    def __len__(self):
        return(len(self.image_paths))

    def setup_transforms(self):
        # Preprocessing transform - Resize. Applied first to reduce memory and time required to apply other transforms.
        transforms = []

        # Augmentation to be applied to the training set.
        if self.is_train:
            transforms.extend([
                A.HorizontalFlip(p=0.5),
                A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=0.2,
                                   shift_limit=0.2, p=0.5, border_mode=0),
            ])

        # Preprocess transforms - Normalization and converting to PyTorch tensor format (HWC --> CHW).
        transforms.extend([
            # The mean and std values are of the IMAGENET dataset and is utilized by every pretrained PyTorch model.
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], always_apply=True),

            ToTensorV2() # (H, W, C) --> (C, H, W)
        ])

        return A.Compose(transforms)

    def load_file(self, file_path, 
                  color_mode,
                  interpolation=cv2.INTER_NEAREST):
        file = cv2.imread(file_path, color_mode)
        file = cv2.resize(file, self.image_size, interpolation=interpolation)
        return file

    def __getitem__(self, index):

        # Get image and mask path.
        image_path = self.image_paths[index]

        # Load image using opencv and convert image format from BGR to RGB and resize.
        image = self.load_file(image_path, 
                               1,
                               interpolation=cv2.INTER_CUBIC)[:,:,::-1]

        if self.mask_paths is not None: # True for Training and validation set.
            # Get mask path.
            mask_path  = self.mask_paths[index]

            # Load mask using opencv and convert image format from BGR to RGB and resize.
            mask = self.load_file(mask_path, 
                                  0,
                                  interpolation=cv2.INTER_NEAREST)
                                  

            # Apply Preprocessing (+ Augmentations) transformations to image-mask pair
            transformed = self.transforms(image=image, mask=mask)
            image, mask = transformed['image'], transformed['mask'].to(torch.long)
            return image, mask

        else: # For Test set.

            # Apply Preprocessing transformations to image.
            transformed = self.transforms(image=image)
            image = transformed['image']

            return image



In [ ]:
id2color = {
    0: (0, 0, 0),       # Background
    1: (192, 128, 128), # Person
    2: (0, 128, 0),     # Bike
    3: (128, 128, 128), # Car
    4: (128, 0, 0),     # Drone
    5: (0, 0, 128),     # Boat
    6: (192, 0, 128),   # Animal
    7: (255, 0, 0),     # Obstacle
    8: (192, 128, 0),   # Construction
    9: (0, 64, 0),      # Vegetation
    10: (128, 128, 0),  # Road
    11: (0, 128, 128)   # Sky
 }

In [ ]:
# Function to convert a single channel mask representation to an RGB mask.
def num_to_rgb(num_arr, color_map=id2color):
    single_layer = np.squeeze(num_arr)
    output = np.zeros(num_arr.shape[:2] + (3,))

    for k in color_map.keys():
        output[single_layer == k] = color_map[k]

    return np.float32(output) / 255.0 # return a floating point array in range [0.0, 1.0]

In [ ]:
# Function to overlay a segmentation map on top of an RGB image.
def image_overlay(image, segmented_image):

    alpha = 1.0 # Transparency for the original image.
    beta  = 0.7 # Transparency for the segmentation map.
    gamma = 0.0 # Scalar added to each sum.

    segmented_image = cv2.cvtColor(segmented_image, cv2.COLOR_RGB2BGR)

    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    image = cv2.addWeighted(image, alpha, segmented_image, beta, gamma, image)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    return np.clip(image, 0.0, 1.0)

In [ ]:
def display_image_and_mask(*, images, masks, color_mask=False, color_map=id2color):
    title = ['GT Image', 'GT Mask']

    for idx in range(images.shape[0]):

        image = images[idx]
        grayscale_gt_mask = masks[idx]

        plt.figure(figsize=(12, 4))

        # Create RGB segmentation map from grayscale segmentation map.
        rgb_gt_mask = num_to_rgb(grayscale_gt_mask, color_map=color_map)

        # Create the overlayed image.
        overlayed_image = image_overlay(image, rgb_gt_mask)

        # Plot image, segmentation map and overlayed image.
        plt.subplot(1, 2, 1)
        plt.title(title[0])
        plt.imshow(image)
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.title(title[1])
        if color_mask:
            plt.imshow(rgb_gt_mask)
        else:
            plt.imshow(grayscale_gt_mask)
        plt.axis('off')

        plt.show()
        plt.close()
    return

In [ ]:
def get_dataloader(batch_size=4, num_workers=0, pin_memory=False):

    NUM_CLASSES = DatasetConfig.NUM_CLASSES

    # OpenCV .resize(..) method accepts new size in format (new_width, new_height).
    IMAGE_SIZE = (DatasetConfig.IMG_WIDTH, DatasetConfig.IMG_HEIGHT)

    # Training image and mask paths.
    train_images = sorted(glob(f"{DatasetConfig.DATA_TRAIN_IMAGES}"))
    train_masks  = sorted(glob(f"{DatasetConfig.DATA_TRAIN_LABELS}"))

    # Validation image and mask paths.
    valid_images = sorted(glob(f"{DatasetConfig.DATA_VALID_IMAGES}"))
    valid_masks  = sorted(glob(f"{DatasetConfig.DATA_VALID_LABELS}"))

    # test images
    test_images = sorted(glob(f"{DatasetConfig.DATA_TEST_IMAGES}"))
    

    # Create training dataset and dataloader.
    train_dataset = CustomSegDataset(
        image_paths=train_images, mask_paths=train_masks, is_train=True,
        num_classes=NUM_CLASSES, image_size=IMAGE_SIZE,
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,  pin_memory=pin_memory,
        num_workers=num_workers, drop_last=True, shuffle=True
    )

    # Create validation dataset and dataloader.
    valid_dataset = CustomSegDataset(
        image_paths=valid_images, mask_paths=valid_masks, is_train=False,
        num_classes=NUM_CLASSES, image_size=IMAGE_SIZE,
    )

    valid_loader = DataLoader(
        valid_dataset, batch_size=batch_size,  pin_memory=pin_memory,
        num_workers=num_workers, shuffle=False
    )

    # create test dataset and dataloader
    test_dataset = CustomSegDataset(
        image_paths=test_images, mask_paths=None, is_train=False,
        num_classes=NUM_CLASSES, image_size=IMAGE_SIZE,
    )

    test_loader = DataLoader(
        test_dataset, batch_size=batch_size,  pin_memory=pin_memory,
        num_workers=num_workers, shuffle=False
    )

    return train_loader, valid_loader, test_loader

In [ ]:
train_loader, valid_loader, test_loader = get_dataloader(batch_size=InferenceConfig.BATCH_SIZE)

## <font style="color:green">1.2. Visualize dataset [3 Points]</font>

**In this sub-section,  you have to plot a few images and its mask.**

**for example:**

---

<img src="https://www.learnopencv.com/wp-content/uploads/2020/04/c3-w12-data-sample.png">

---

In [ ]:
# This function is used for reversing the Normalization step performed during image preprocessing.
# Note the mean and std values must match the ones used.

def denormalize(tensors, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Normalization parameters for pre-trained PyTorch models
    Denormalizes image tensors using mean and std"""

    for c in range(3):
        tensors[:, c, :, :].mul_(std[c]).add_(mean[c])

    return torch.clamp(tensors, min=0.0, max=1.0)

In [ ]:
for batch_images, batch_masks in valid_loader:

    batch_images = denormalize(batch_images).permute(0, 2, 3, 1).numpy()
    batch_masks  = batch_masks.numpy()

    display_image_and_mask(images=batch_images, masks=batch_masks, color_mask=True)

    break

# <font style="color:green">2. Evaluation Metrics [10 Points]</font>

<p>This competition is evaluated on the mean <a href='https://en.wikipedia.org/wiki/Sørensen–Dice_coefficient'>Dice coefficient</a
>. The Dice coefficient can be used to compare the pixel-wise agreement between a predicted segmentation and its corresponding ground truth. The formula is given by: </p>

<p>$$DSC =  \frac{2 |X \cap Y|}{|X|+ |Y|}$$
$$ \small \mathrm{where}\ X = Predicted\ Set\ of\ Pixels,\ \ Y = Ground\ Truth $$ </p>
<p>The Dice coefficient is defined to be 1 when both X and Y are empty.</p>

**In this section, you have to implement the dice coefficient evaluation metric.**

In [ ]:
def dice_mean_metric(predictions, ground_truths, num_classes=12, dims = (1, 2), smooth=1e-8):
    """
    Arguments:
    predictions (torch.tensor): Prediction (P) model output logits.
                                Shape: [batch_size(B), num_classes, height(H), width(W)]

    ground_truths (torch.tensor): Ground truth mask (G). [B, num_classes, H, W].

    dims (tuple): Dimensions corresponding to image height and width in a tensor shape: [B, H, W, num_classes].

    Returns:
    torch.tensor: A scalar tensor representing the Mean Dice coefficient.
    """

    # Convert single channel ground truth masks into one-hot encoded tensor.
    # Shape: (B, H, W, num_classes)
    ground_truth_oh = F.one_hot(ground_truths, num_classes=num_classes)

    # Normalize model predictions and transpose.
    # Shape :: [B, num_classes, H, W] --> [B, H, W, num_classes]
    # This is done to match the shape of ground_truth_oh.
    prediction_norm = F.softmax(predictions, dim=1).permute(0, 2, 3, 1)

    # Intersection: |G ∩ P|. Shape: [B, num_classes]
    intersection = (prediction_norm * ground_truth_oh).sum(dim=dims)

    # Summation: |G| + |P|. Shape: [B, num_classes].
    summation = (prediction_norm.sum(dim=dims) + ground_truth_oh.sum(dim=dims))

    # Dice Shape: [B, num_classes]
    dice = (2.0 * intersection + smooth) / (summation + smooth)

    # Compute the mean over the remaining axes (batch and classes).
    dice_mean = dice.mean()

    return dice_mean


In [ ]:
def dice_coef_loss(predictions, ground_truths, num_classes=12, dims = (1, 2), smooth=1e-8):
    """
    Calculate a combined loss function comprising the Naive Dice coefficient loss and cross-entropy loss
    Arguments:
    predictions (torch.tensor): Prediction (P) model output logits.
                                Shape: [batch_size(B), num_classes, height(H), width(W)]

    ground_truths (torch.tensor): Ground truth mask (G). [B, num_classes, H, W].

    dims (tuple): Dimensions corresponding to image height and width in a tensor shape: [B, H, W, num_classes].

    Returns:
    torch.tensor: A scalar tensor representing the Mean Dice coefficient.
    """

    # Convert single channel ground truth masks into one-hot encoded tensor.
    # Shape: (B, H, W, num_classes)
    ground_truth_oh = F.one_hot(ground_truths, num_classes=num_classes)

    # Normalize model predictions and transpose.
    # Shape :: [B, num_classes, H, W] --> [B, H, W, num_classes]
    # This is done to match the shape of ground_truth_oh.
    prediction_norm = F.softmax(predictions, dim=1).permute(0, 2, 3, 1)

    # Intersection: |G ∩ P|. Shape: [B, num_classes]
    intersection = (prediction_norm * ground_truth_oh).sum(dim=dims)

    # Summation: |G| + |P|. Shape: [B, num_classes].
    summation = (prediction_norm.sum(dim=dims) + ground_truth_oh.sum(dim=dims))

    # Dice Shape: [B, num_classes]
    dice = (2.0 * intersection + smooth) / (summation + smooth)

    # Compute the mean over the remaining axes (batch and classes).
    dice_mean = dice.mean()

    # Compute cross-entropy loss.
    CE = F.cross_entropy(predictions, ground_truths)

    return (1.0 - dice_mean) + CE

In [ ]:
def mean_iou(predictions, ground_truths, num_classes=12, dims = (1, 2)):
    """
    Arguments:
    predictions (torch.tensor): Prediction (P) from the model with or without softmax.
                                Shape: [batch_size(B), height(H), width(W)]

    ground_truths (torch.tensor): Ground truth mask (G). Shape: [B, H, W]

    dims (tuple): Dimensions corresponding to image height and width in a tensor shape: [B, H, W, C].

    Returns:
    torch.tensor: A scalar tensor representing the Classwise Mean IoU metric.
    """

    # Convert single channel ground truth masks into one-hot encoded tensor.
    # Shape: [B, H, W] --> [B, H, W, num_classes]
    ground_truths = F.one_hot(ground_truths, num_classes=num_classes)

    # Converting unnormalized predictions into one-hot encoded across channels.
    # Shape: [B, H, W] --> [B, H, W, num_classes]
    predictions = F.one_hot(predictions, num_classes=num_classes)

    # Intersection: |G ∩ P|. Shape: [B, num_classes]
    intersection = (predictions * ground_truths).sum(dim=dims)

    # Summation: |G| + |P|. Shape: [B, num_classes].
    summation = (predictions.sum(dim=dims) + ground_truths.sum(dim=dims))

    # Union. Shape: [B, num_classes]
    union = summation - intersection

    # IoU Shape: [B, num_classes]
    iou = intersection / union

    # As no smoothing is used we replace any 'nan' value that with 0.
    # With smoothing the results yields slightly different values.
    iou = torch.nan_to_num(iou, nan=0.0)

    # Iou for each class
    per_class_iou = iou.mean(dim=0)

    # Shape: [batch_size,]
    num_classes_present = torch.count_nonzero(summation, dim=1)

    # IoU per image.
    # Average over the total number of classes present in ground_truths and predictions.
    # Shape: [batch_size,]
    iou = iou.sum(dim=1) / num_classes_present

    # Compute the mean over the remaining axes (batch and classes).
    # Shape: Scalar
    iou_mean = iou.mean()

    return iou_mean, per_class_iou


# <font style="color:green">3. Model [10 Points]</font>

**In this section, you have to define your model.**

In [ ]:
@dataclass
class ModelConfig:
    MODEL_NAME:             str = "deeplabv3_resnet50"
    USE_PRETRAINED:        bool = True
    AUX_LOSS: Union[bool, None] = True # True, None
    AUX_WEIGHT:           float = 0.5

In [ ]:
def prepare_model(model_name="deeplabv3_resnet50", use_pretrained=True, num_classes=2):
    WEIGHTS = "DEFAULT" if use_pretrained else None

    try:
        print(f"Loading model: {model_name}, Use pretrained? {use_pretrained}")
        model = getattr(segmentation, model_name.lower())(weights=WEIGHTS, aux_loss=ModelConfig.AUX_LOSS)

    except Exception as e:
        print(e)
        print("\nLoading pretrained DeepLabV3 model with a resnet50 backbone.")
        model = segmentation.deeplabv3_resnet50(weights="DEFAULT")


    if "lraspp" in model_name.lower():
        # Updating the out_channels of the low and high classifer layers in the LRASPP model.
        model.low_classifier  = nn.Conv2d(in_channels=40, out_channels=num_classes, 
                                          kernel_size=1, stride=1)
        model.high_classifier = nn.Conv2d(in_channels=128, out_channels=num_classes, 
                                          kernel_size=1, stride=1)
    else:
        # Update the number of output channels to num_classes in the final layer in the main path.
        model.classifier[-1] = nn.LazyConv2d(out_channels=num_classes, 
                                             kernel_size=1, 
                                             stride=1)

        if use_pretrained: # Auxilary branch is absent if pretrained model is not loaded.

            if ModelConfig.AUX_LOSS:
                # Update the number of output channels to num_classes in the final layer of the auxiliary branch.
                model.aux_classifier[-1] = nn.LazyConv2d(out_channels=num_classes, 
                                                         kernel_size=1, 
                                                         stride=1)
            else:
                # Removing auxiliary classifier block.
                model.aux_classifier = nn.Identity()
    return model

In [ ]:
model = prepare_model(
    model_name=ModelConfig.MODEL_NAME,
    use_pretrained=ModelConfig.USE_PRETRAINED,
    num_classes=DatasetConfig.NUM_CLASSES,
)
model.eval()

summary(model, input_size=(1, 3, DatasetConfig.IMG_HEIGHT, DatasetConfig.IMG_WIDTH))

# <font style="color:green">4. Train & Inference</font>

- **In this section, you have to train the model and infer on sample data.**


- **You can write your trainer class in this section.**


- **If you are using any loss function other than PyTorch standard loss function, you have to define in this section.**


- **This section should also have optimizer and LR-schedular (if using) details.**



In [ ]:
def train_one_epoch(
    model, loader,
    optimizer, scaler,
    num_classes, device="cpu",
    epoch_idx= 800, total_epochs=50,
    scheduler = None
):

    # Change model mode.
    model.train()

    loss_record   = MeanMetric()
    metric_record = MeanMetric()
    acc_record    = MulticlassAccuracy(num_classes=num_classes, 
                                       average="micro")

    loader_len = len(loader)

    with tqdm(total=loader_len, ncols=122) as tq:
        tq.set_description(f"Train :: Epoch: {epoch_idx}/{total_epochs}")

        for data, target in loader:
            tq.update(1)

            # Send data and target to GPU device if available.
            data, target = data.to(device), target.to(device)

            # Reset parameters gradient to zero.
            optimizer.zero_grad()

            with amp.autocast(): # Autocasting for mixed-precision training.
                # Perform Forward pass through the model. Output is a dictionary.
                output_dict = model(data)

                cls_out = output_dict['out'] # Classifier head output

                # Calculate Combo loss (Segmentation specific loss (Dice) + cross entropy)
                loss = dice_coef_loss(cls_out, 
                                      target, 
                                      num_classes=num_classes)

                if ModelConfig.AUX_LOSS:
                    aux_out = output_dict['aux'] # Auxiliary head output

                    # Calculate simple cross entropy loss for auxiliary head output.
                    aux_loss = F.cross_entropy(aux_out, 
                                               target)

                    # A weighted combination between classifier and auxiliary head loss.
                    loss += ModelConfig.AUX_WEIGHT * aux_loss

            # Calculate gradients w.r.t training parameters on scaled loss.
            scaler.scale(loss).backward()

            # Update parameters using gradients
            scaler.step(optimizer)

            # Updates the scale for the next iteration.
            scaler.update()

            # scheduler
            if scheduler:
                scheduler.step()

            # Detach Classifier head output logits tensor from graph.
            cls_out = cls_out.detach()

            # Get the index across channel axis of the max logit score.
            # We can directly call argmax because: Softmax(logit).argmax() == logit.argmax()
            pred_idx = cls_out.argmax(dim=1)

            # Calculate Segmentation specific metric (Dice).
            metric = dice_mean_metric(cls_out, 
                                      target, 
                                      num_classes=num_classes)
            
            # Record and calculate batch accuracy using torchmetrics.
            acc_record.update(pred_idx.cpu(), 
                              target.cpu())

            # Record loss and IoU metric.
            loss_record.update(loss.detach().cpu(), 
                               weight=data.shape[0])
            metric_record.update(metric.cpu(),     
                                 weight=data.shape[0])

            # Update progress bar description.
            tq.set_postfix_str(s=f"Loss: {loss_record.compute():.4f}, IoU: {metric_record.compute():.4f}, Acc: {acc_record.compute():.4f}")

    # Get mean loss, accuracy and IoU score.
    epoch_loss   = loss_record.compute()
    epoch_metric = metric_record.compute()
    epoch_acc    = acc_record.compute()

    return epoch_loss, epoch_metric, epoch_acc

In [ ]:
def validate(
    model, loader,
    device, num_classes,
    epoch_idx, total_epochs,
):

    # Change model mode.
    model.eval()

    loss_record   = MeanMetric()
    metric_record = MeanMetric()
    acc_record    = MulticlassAccuracy(num_classes=num_classes, 
                                       average="micro")

    loader_len = len(loader)

    # A list to accumulate the 12 class scores across all batches
    class_scores_accumulator = []

    with tqdm(total=loader_len, ncols=122) as tq:
        tq.set_description(f"Valid :: Epoch: {epoch_idx}/{total_epochs}")

        for data, target in loader:
            tq.update(1)

            # Send data and target to GPU device if available.
            data, target = data.to(device), target.to(device)

            with torch.no_grad():
                # Perform Forward pass through the model. Output is a dictionary.
                output_dict = model(data)

            cls_out = output_dict['out'] # Classifier head output

            # Calculate Combo loss (Segmentation specific loss (Dice) + cross entropy)
            loss = dice_coef_loss(cls_out, 
                                  target, 
                                  num_classes=num_classes)

            # Get the index across channel axis of the max logit score.
            # We can directly call argmax because: Softmax(logit).argmax() == logit.argmax()
            pred_idx = cls_out.argmax(dim=1)

            # Calculate Segmentation specific metric (Dice).
            metric = dice_mean_metric(cls_out, 
                                      target, 
                                      num_classes=num_classes)

            # Iou for all 12 classes
            _, per_class_metrics = mean_iou(pred_idx, 
                                            target, 
                                            num_classes=num_classes)

            # Store the 12 individual values
            class_scores_accumulator.append(per_class_metrics.cpu())

            # Record and calculate batch accuracy using torchmetrics.
            acc_record.update(pred_idx.cpu(), 
                              target.cpu())

            # Record loss and IoU metric.
            loss_record.update(loss.cpu(),     
                               weight=data.shape[0])
            metric_record.update(metric.cpu(), 
                                 weight=data.shape[0])

        # Compute Epoch loss, accuracy and IoU score.
        valid_epoch_loss   = loss_record.compute()
        valid_epoch_metric = metric_record.compute()
        valid_epoch_acc    = acc_record.compute()

        # Calculate the final average for each of the 12 classes across all batches
        final_per_class = torch.stack(class_scores_accumulator).mean(dim=0)

        # Update progress bar description to display epoch log.
        tq.set_postfix_str(s=f"Valid Loss: {valid_epoch_loss:.4f}, Valid IoU: {valid_epoch_metric:.4f}, Valid Acc: {valid_epoch_acc:.4f}")

    return valid_epoch_loss, valid_epoch_metric, valid_epoch_acc, final_per_class

In [ ]:
def main(*, model, 
         optimizer, ckpt_dir, 
         pin_memory=False, device="cpu", 
         scheduler_dict = None):

    # Create Dataloader.
    train_loader, valid_loader,_ = get_dataloader(batch_size=TrainingConfig.BATCH_SIZE, 
                                                  pin_memory=pin_memory,
                                                  num_workers=TrainingConfig.NUM_WORKERS)

    # Save the model if best_loss improves.
    best_loss = float("inf")

    # Plot training and validation epoch logs and
    # prints min, max, current stats of all variable being tracked.
    live_plot = PlotLosses(outputs=[MatplotlibPlot(cell_size=(8, 3)), ExtremaPrinter()])

    total_epochs = TrainingConfig.EPOCHS

    # Creates a GradScaler once at the beginning of training
    # for automatic mixed precision training.
    scaler = torch.amp.GradScaler()

    # unpack the scheduler if scheduler is used
    if scheduler_dict:
        scheduler = scheduler_dict['scheduler']
        update_per_epoch = scheduler_dict['update_per_epoch']
    else:
        scheduler = None
        update_per_epoch = None

    # Training Loop.
    for epoch in range(total_epochs):

        # Memory Cleanup.
        torch.cuda.empty_cache()
        gc.collect()

        # Train one epoch.
        train_loss, train_metric, train_acc = train_one_epoch(model=model, loader=train_loader, 
                                                              optimizer=optimizer, scaler=scaler,
                                                              num_classes=DatasetConfig.NUM_CLASSES, device=device, 
                                                              epoch_idx=epoch+1, total_epochs=total_epochs,
                                                              scheduler = None if update_per_epoch else scheduler)

        # Peform model validation.
        valid_loss, valid_metric, valid_acc, Iou_per_class = validate(model=model, loader=valid_loader, 
                                                                      device=device, num_classes=DatasetConfig.NUM_CLASSES,
                                                                      epoch_idx=epoch+1, total_epochs=total_epochs)

        if scheduler and update_per_epoch:
            scheduler.step()

        # Plot train and validation statistics.
        plot_data = {
            "loss": train_loss,
            "val_loss": valid_loss,
            "accuracy": train_acc,
            "val_accuracy": valid_acc,
            "dice_train_metric":train_metric,
            "dice_val_metric": valid_metric,
        }

        # Add 12 Classes as separate top-level keys
        for i, score in enumerate(Iou_per_class):
            plot_data[f"IoU_Class_{i}"] = score.item()

        # 3. Update and send
        live_plot.update(plot_data)
        live_plot.send()

        # Create model and optimizer checkpoint.
        if valid_loss < best_loss:
            best_loss = valid_loss
            print("Model Improved. Saving...", end="")

            checkpoint_dict = {
                "opt": optimizer.state_dict(),
                "model": model.state_dict(),
                "scaler": scaler.state_dict(),
            }
            torch.save(checkpoint_dict, os.path.join(ckpt_dir, "ckpt.tar"))
            del checkpoint_dict
            print("Done.\n")

    return

In [ ]:
def create_checkpoint_dir(checkpoint_dir):
    # Create a new checkpoint directory every time.
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    try:
        num_versions = [int(i.split("_")[-1]) for i in os.listdir(checkpoint_dir) if "version" in i]
        version_num = max(num_versions) + 1
    except:
        version_num = 0

    version_dir = os.path.join(checkpoint_dir, "version_" + str(version_num))
    os.makedirs(version_dir)

    print(f"Checkpoint directory: {version_dir}")
    return version_dir

## <font style="color:green">4.1. Train [7 Points]</font>

**Write your training code in this sub-section.**


**This section must contain training plots (use matplotlib or share tensorboard.dev scalars logs).**

**You must have to plot the following:**
- **train loss**


- **validation loss**


- **IoU for all twelve classes (0-11) and the mean IoU of all classes on validatin data.** 

**an example of matplotlib plot:**

---

<img src='https://www.learnopencv.com/wp-content/uploads/2020/04/c3-w12-train-loss.png'>

---

<img src='https://www.learnopencv.com/wp-content/uploads/2020/04/c3-w12-val-loss.png'>

---

<img src='https://www.learnopencv.com/wp-content/uploads/2020/04/c3-w12-mean_iou.png'>

---

<img src='https://www.learnopencv.com/wp-content/uploads/2020/04/c3-w12-iou-0.png'>

---

<center>*</center>
<center>*</center>
<center>*</center>

---

<img src='https://www.learnopencv.com/wp-content/uploads/2020/04/c3-w12-iou-11.png'>

---


In [ ]:
# For deterministic training
seed_everything(seed_value=41)

# Set default device to GPU if available.
DEVICE, GPU_AVAILABLE = get_default_device()

# Setup Model Checkpoint directory.
CKPT_DIR = create_checkpoint_dir(TrainingConfig.CHECKPOINT_DIR)

# Create model.
model = prepare_model(
    model_name=ModelConfig.MODEL_NAME,
    use_pretrained=ModelConfig.USE_PRETRAINED,
    num_classes=DatasetConfig.NUM_CLASSES,
)

# Send model to device (GPU/CPU)
model.to(DEVICE)

# Dummy pass through the model to initalize parameters.
_ = model(torch.randn((2, 3, DatasetConfig.IMG_HEIGHT, DatasetConfig.IMG_WIDTH), device=DEVICE))

# Initialize Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=TrainingConfig.LEARNING_RATE,
    weight_decay=1e-4,
)

# Initialize learning rate scheduler
scheduler = OneCycleLR(
    optimizer,
    max_lr = 1e-3,
    steps_per_epoch=len(train_loader),
    epochs=TrainingConfig.EPOCHS,
    pct_start=0.3,        # warmup phase (30%)
    anneal_strategy='cos',
    div_factor=10,        # initial lr = max_lr / div_factor
    final_div_factor=100
)

scheduler_dict = {
    "scheduler":scheduler,
    "update_per_epoch":False
}
# Compile model if you are using PyTorch 2.0 for RTX 30XX series and above.
# model = torch.compile(model)

In [ ]:
main(
    model=model,
    optimizer=optimizer,
    ckpt_dir=CKPT_DIR,
    device=DEVICE,
    pin_memory=GPU_AVAILABLE,
    scheduler_dict = scheduler_dict
)

## <font style="color:green">4.2. Inference [3 Points]</font>

**Plot some sample inference in this sub-section.**

**for example:**

---

<img src='https://www.learnopencv.com/wp-content/uploads/2020/04/c3-w12-sample-predtiction.png'>

---



## Load the trained model

In [ ]:
DEVICE, GPU_AVAILABLE = get_default_device()

CKPT_DIR = os.path.join(TrainingConfig.CHECKPOINT_DIR, 'version_0')

trained_model = prepare_model(
    model_name=ModelConfig.MODEL_NAME,
    use_pretrained=ModelConfig.USE_PRETRAINED,
    num_classes=DatasetConfig.NUM_CLASSES,
)

# Initialize parameters of the model before compiling.
# This step is necessary if model contains "Lazy" modules.
_ = trained_model(torch.randn(2, 3, DatasetConfig.IMG_HEIGHT, DatasetConfig.IMG_WIDTH))

# trained_model = torch.compile(trained_model)
trained_model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "ckpt.tar"), map_location='cpu')['model'])

trained_model.to(DEVICE)
trained_model.eval();

In [ ]:
@torch.inference_mode()
def inference(model, loader, device="cpu"):
    num_batches_to_process = InferenceConfig.NUM_BATCHES

    for idx, (batch_img, batch_mask) in enumerate(loader):
        pred_all = model(batch_img.to(device))['out']
        pred_all = pred_all.cpu().argmax(dim=1).numpy()

        batch_img = denormalize(batch_img.cpu()).permute(0, 2, 3, 1).numpy()

        if idx == num_batches_to_process:
            break

        for i in range(0, len(batch_img)):
            fig = plt.figure(figsize=(20, 8))

            # Display the original image.
            ax1 = fig.add_subplot(1, 4, 1)
            ax1.imshow(batch_img[i])
            ax1.title.set_text("Actual frame")
            plt.axis("off")

            # Display the ground truth mask.
            true_mask_rgb = num_to_rgb(batch_mask[i], color_map=id2color)
            ax2 = fig.add_subplot(1, 4, 2)
            ax2.set_title("Ground truth labels")
            ax2.imshow(true_mask_rgb)
            plt.axis("off")

            # Display the predicted segmentation mask.
            pred_mask_rgb = num_to_rgb(pred_all[i], color_map=id2color)
            ax3 = fig.add_subplot(1, 4, 3)
            ax3.set_title("Predicted labels")
            ax3.imshow(pred_mask_rgb)
            plt.axis("off")

            plt.show()

In [ ]:
inference(trained_model, valid_loader, device=DEVICE)

# <font style="color:green">5. Prepare Submission CSV [10 Points]</font>

**Write your code to prepare the submission CSV file.**


**Note that in the submission file, you have to write Encoded Pixels.**

[Here is a blog to understand what is Encoded Pixels.](https://medium.com/analytics-vidhya/generating-masks-from-encoded-pixels-semantic-segmentation-18635e834ad0)

### performing inference on test dataset

In [ ]:
@torch.inference_mode()
def prepare_submission(model, loader, device):
    model.eval()
    submission_data = []
    
    # To Ensure this list is sorted identically to how the loader processes files
    test_image_paths = sorted(glob(f"{DatasetConfig.DATA_TEST_IMAGES}"))
    img_idx = 0

    for batch in tqdm(loader, desc="Generating RLE"):
        # Handle batch unpacking
        if isinstance(batch, (list, tuple)):
            batch_img = batch[0].to(device)
        else:
            batch_img = batch.to(device)
        
        # Forward pass
        outputs = model(batch_img)['out']
        # preds shape: [Batch, H_model, W_model]
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for i in range(len(preds)):
            # Get the base filename without extension
            if img_idx < len(test_image_paths):
                base_name = os.path.basename(test_image_paths[img_idx]).split('.')[0]
            else:
                base_name = f"unknown_{img_idx}"

            # --- KEY MODIFICATION: Iterate through all 12 classes ---
            for class_id in range(12):
                # 1. Create unique IMG_ID for this class (e.g., image01_0)
                img_id_with_class = f"{base_name}_{class_id}"
                
                # 2. Extract binary mask for the specific class
                binary_mask = (preds[i] == class_id).astype(np.uint8)
                
                # 3. Resize to original dimensions (1280, 720)
                resized_mask = cv2.resize(
                    binary_mask, 
                    (1280, 720), 
                    interpolation=cv2.INTER_NEAREST
                )
                
                # 4. Encode to RLE 
                rle_string = rle_encode(resized_mask)
                
                submission_data.append({
                    "ImageID": img_id_with_class,
                    "EncodedPixels": rle_string
                })
            
            img_idx += 1
            
    # Returning with column names
    return pd.DataFrame(submission_data, columns=['ImageID', 'EncodedPixels'])

In [ ]:
def rle_encode(mask):
    """
    mask: numpy array, 1 - mask, 0 - background
    Returns run length as string formated
    """
    # Flatten in column-major order (Fortran order) 
    # to match "top to bottom, then left to right"
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

submission_df = prepare_submission(trained_model, test_loader, device=DEVICE)

# Save the file
submission_df.to_csv("submission.csv", index=False)

print("\n--- SUBMISSION PREVIEW ---")
print(submission_df.head())
print(f"\nTotal rows: {len(submission_df)}")

# <font style="color:green">6. Kaggle Profile Link [50 Points]</font>

Share your Kaggle profile link here with us so that we can give points for the competition score. 

You should have a minimum dice score of `0.60` on the test data to get all points. If the dice score is less than `0.55`, you will not get any points for the section. 

**You must have to submit `submission.csv` (prediction for images in `test.csv`) in `Submit Predictions` tab in Kaggle to get any evaluation in this section.**

In [ ]:
https://www.kaggle.com/suryasai99